# 🧪 UAT Environment Pipeline
**Branch:** `uat` | **DB:** `UAT_DB` | **Triggers:** PR merged from `develop → uat`

### What this notebook does
1. Validate UAT_DB environment
2. Clone latest DEV_DB snapshot → UAT_DB (zero-copy)
3. Run integration tests (stricter than DEV)
4. Run data quality gates
5. Generate UAT sign-off report
6. Log deployment — if passed, auto-create PR to main

In [ ]:
# ── CELL 1: Init
import time, json
from snowflake.snowpark.context import get_active_session

session = get_active_session()
ENV     = 'UAT'
DB      = 'UAT_DB'
SRC_DB  = 'DEV_DB'
start   = time.time()
results = {}

print(f'🧪 {ENV} Pipeline Starting')
print(f'   DB      : {session.get_current_database()}')
print(f'   Role    : {session.get_current_role()}')
print(f'   WH      : {session.get_current_warehouse()}')

In [ ]:
# ── CELL 2: Zero-Copy Clone DEV → UAT (point-in-time snapshot)
print('Cloning DEV_DB → UAT_DB (zero-copy)...')

clone_ts = session.sql("SELECT CURRENT_TIMESTAMP()").collect()[0][0]

for schema in ['RAW', 'STAGING', 'ANALYTICS']:
    try:
        # Drop old UAT schema and clone fresh from DEV
        session.sql(f"DROP SCHEMA IF EXISTS {DB}.{schema}").collect()
        session.sql(f"""
            CREATE SCHEMA {DB}.{schema}
            CLONE {SRC_DB}.{schema}
        """).collect()
        # Count tables cloned
        count = session.sql(f"""
            SELECT COUNT(*) FROM {DB}.INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = '{schema}'
        """).collect()[0][0]
        print(f'  ✓ {DB}.{schema} cloned ({count} tables)')
    except Exception as e:
        print(f'  ~ {schema}: {e}')

results['clone'] = f'DEV_DB → UAT_DB at {clone_ts}'
print('Clone complete.')

In [ ]:
# ── CELL 3: Integration Tests (stricter than DEV unit tests)
print('Running UAT integration tests...')

def test(name, sql, check, critical=True):
    try:
        val = session.sql(sql).collect()[0][0]
        ok  = check(val)
        print(f"  {'✓' if ok else ('✗' if critical else '!')} {name}: {val}")
        return ok
    except Exception as e:
        print(f'  ✗ {name}: ERROR — {e}')
        return not critical

critical_fails = 0

# Data volume gates
critical_fails += not test('Min customers (≥5)',   f'SELECT COUNT(*) FROM {DB}.RAW.CUSTOMERS', lambda v: v >= 5)
critical_fails += not test('Min orders (≥10)',     f'SELECT COUNT(*) FROM {DB}.RAW.ORDERS', lambda v: v >= 10)

# Data quality gates
critical_fails += not test('No null customer_id', f'SELECT COUNT(*) FROM {DB}.RAW.ORDERS WHERE customer_id IS NULL', lambda v: v == 0)
critical_fails += not test('No negative amounts', f'SELECT COUNT(*) FROM {DB}.RAW.ORDERS WHERE amount < 0', lambda v: v == 0)
critical_fails += not test('No future order dates',f"SELECT COUNT(*) FROM {DB}.RAW.ORDERS WHERE order_date > CURRENT_DATE()", lambda v: v == 0)

# Referential integrity
critical_fails += not test('Orders→Customers RI', f"""
    SELECT COUNT(*) FROM {DB}.RAW.ORDERS o
    WHERE NOT EXISTS (SELECT 1 FROM {DB}.RAW.CUSTOMERS c WHERE c.customer_id = o.customer_id)
""", lambda v: v == 0)

# Business logic
critical_fails += not test('Completed orders > 0', f"SELECT COUNT(*) FROM {DB}.RAW.ORDERS WHERE status='completed'", lambda v: v > 0)

# Non-critical warnings
test('Cancelled rate < 50%', f"SELECT RATIO_TO_REPORT(COUNT_IF(status='cancelled')) OVER() * 100 FROM {DB}.RAW.ORDERS LIMIT 1", lambda v: v < 50, critical=False)

results['integration_tests'] = f'{7 - critical_fails}/7 passed'
if critical_fails > 0:
    raise Exception(f'UAT integration tests FAILED: {critical_fails} critical failures')
print('All UAT tests passed.')

In [ ]:
# ── CELL 4: UAT Sign-off Report
print('Generating UAT sign-off report...')

from snowflake.snowpark.functions import col, count, sum as sum_, avg, max as max_

df_orders = session.table(f'{DB}.RAW.ORDERS')
summary = df_orders.group_by('REGION', 'STATUS').agg(
    count('*').alias('ORDER_COUNT'),
    sum_('AMOUNT').alias('TOTAL_REVENUE'),
    avg('AMOUNT').alias('AVG_ORDER_VALUE')
).sort('REGION', 'STATUS')

print('\n  UAT Data Summary:')
print(f'  {"REGION":<10} {"STATUS":<12} {"ORDERS":>8} {"REVENUE":>12} {"AVG ORDER":>12}')
print(f'  {"-"*56}')
for row in summary.collect():
    print(f'  {row[0]:<10} {row[1]:<12} {row[2]:>8} {row[3]:>12.2f} {row[4]:>12.2f}')

results['uat_report'] = 'Generated'
print('\n  ✓ UAT sign-off report generated')

In [ ]:
# ── CELL 5: Log + Auto-promote signal
duration = round(time.time() - start, 1)
results['duration_sec'] = duration

session.sql(f"""
    INSERT INTO UAT_DB.ANALYTICS.DEPLOY_LOG
        (environment, branch, status, duration_sec, notes)
    VALUES ('UAT', 'uat', 'SUCCESS', {duration}, '{json.dumps(results)}')
""").collect()

print()
print('=' * 50)
print(f'  🧪 UAT DEPLOY COMPLETE  ({duration}s)')
print('=' * 50)
for k, v in results.items():
    print(f'  {k:24s}: {v}')
print()
print('  ✅ UAT PASSED — ready for production!')
print('  Next: create a PR from uat → main for PROD deploy')